Logistic Regression:


Contributions:
1. https://www.ibm.com/think/topics/logistic-regression
2. https://www.analyticsvidhya.com/blog/2023/01/a-comprehensive-guide-to-ols-regression-part-1/

In [6]:
# Libraries:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix
)
import os


In [ ]:
# Data import and pre-processing:

os.chdir('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/data_input')

# Data import:
df_factor = pd.read_csv("momentum_labeled.csv", parse_dates=['Date'])
df_factor = df_factor[['Date','trend_bin']]    # trend_bin = trend label
df_factor = df_factor.set_index('Date')

df_features = pd.read_csv("top50_features.csv", parse_dates=['Date'])
df_features = df_features.set_index('Date')


data = pd.concat([df_features,df_factor], axis=1, join='inner')
data = data.dropna()

print(data.columns)

Index(['Momentum_vs_Value_trend_lag1', 'Momentum_relative_volatility_lag1',
       'Momentum_vs_Quality_lag1', 'SA_NB_Curvature_lag1',
       'Value_vs_Quality_trend_lag1', 'SA_NB_Level_TS_lag1',
       'SA_NB_Curvature_QS_lag1', 'Value_excess_kurtosis_lag1',
       'Value_excess_skewness_lag1', 'US_NB_Slope_QS_lag1',
       'Momentum_relative_kurtosis_lag1', 'Value_vs_Quality_Cumulative_lag1',
       'US_NB_Level_TS_lag1', 'SA_NB_Slope_QS_lag1',
       'Quality_momentum_12M_lag1', 'Momentum_excess_momentum_lag1',
       'Momentum_vs_Quality_trend_lag1', 'Momentum_momentum_1M_lag1',
       'Quality_relative_skewness_lag1', 'Momentum_excess_skewness_lag1',
       'trend_bin'],
      dtype='object')


In [8]:

class LogisticRegression:
    def __init__(self, learning_rate=0.01, num_iterations=1000, tol=1e-4):
        self.learning_rate = learning_rate
        self.num_iterations = num_iterations
        self.tol = tol
        self.beta = None
        self.cost_history = []
    
    def sigmoid(self, z):
        # Stable sigmoid
        return np.where(z >= 0,
                        1 / (1 + np.exp(-z)),
                        np.exp(z) / (1 + np.exp(z)))
    
    def cost_function(self, X, y, beta):
        m = len(y)
        h = self.sigmoid(X.dot(beta))
        # To avoid log(0)
        h = np.clip(h, 1e-10, 1 - 1e-10)
        cost = (-1/m) * np.sum(y * np.log(h) + (1-y) * np.log(1-h))
        return cost
    
    def gradient_descent(self, X, y, beta):
        m = len(y)
        for i in range(self.num_iterations):
            h = self.sigmoid(X.dot(beta))
            gradient = (1/m) * X.T.dot(h - y)
            new_beta = beta - self.learning_rate * gradient
            
            # Check for convergence
            if np.linalg.norm(new_beta - beta) < self.tol:
                print(f"Converged after {i} iterations")
                break
                
            beta = new_beta
            self.cost_history.append(self.cost_function(X, y, beta))
        return beta
    
    def fit(self, X, y):
        # Add intercept term and initialize beta
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        self.beta = np.zeros(X_b.shape[1])
        self.beta = self.gradient_descent(X_b, y, self.beta)
    
    def predict_proba(self, X):
        # Returns probability estimates
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return self.sigmoid(X_b.dot(self.beta))
    
    def predict(self, X, threshold=0.5):
        # Returns class labels
        return (self.predict_proba(X) >= threshold).astype(int)



In [9]:
def walk_forward_validation(X, y, model_class, initial_train_size, test_size=1, **model_params):
    n_samples = len(y)
    predictions = []
    probas = []
    true_values = []

    train_end = initial_train_size
    while train_end + test_size <= n_samples:
        # Split
        X_train, y_train = X[:train_end], y[:train_end]
        X_test, y_test = X[train_end:train_end+test_size], y[train_end:train_end+test_size]

        # Train
        model = model_class(**model_params)
        model.fit(X_train, y_train)

        # Predict
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        predictions.extend(y_pred)
        probas.extend(y_proba)
        true_values.extend(y_test)

        # Move forward
        train_end += test_size

    return np.array(predictions), np.array(probas), np.array(true_values)



In [10]:
# Run regression

if __name__ == "__main__":
    # Load data
    X = data.iloc[:, :-2].values
    y = data.iloc[:, -1].values
    X = np.nan_to_num(X, nan=np.nan, posinf=1e10, neginf=-1e10)  # Convert inf to large finite numbers

    preds, probas, true_vals = walk_forward_validation(
        X, y, LogisticRegression,
        initial_train_size=30,
        test_size=1,
        learning_rate=0.1,
        num_iterations=500
    )

    # Evaluation metrics
    acc = accuracy_score(true_vals, preds)
    prec = precision_score(true_vals, preds, zero_division=0)
    rec = recall_score(true_vals, preds, zero_division=0)
    f1 = f1_score(true_vals, preds, zero_division=0)
    roc = roc_auc_score(true_vals, probas)
    ll = log_loss(true_vals, probas)
    cm = confusion_matrix(true_vals, preds)

    print("=== Walk-Forward Evaluation ===")
    print(f"Accuracy    : {acc:.4f}")
    print(f"Precision   : {prec:.4f}")
    print(f"Recall      : {rec:.4f}")
    print(f"F1 Score    : {f1:.4f}")
    print(f"ROC AUC     : {roc:.4f}")
    print(f"Log Loss    : {ll:.4f}")
    print("Confusion Matrix:")
    print(cm)

C:\Users\p528552\AppData\Local\Temp\ipykernel_33268\1788851555.py:13: RuntimeWarning: overflow encountered in exp
  np.exp(z) / (1 + np.exp(z)))
C:\Users\p528552\AppData\Local\Temp\ipykernel_33268\1788851555.py:13: RuntimeWarning: invalid value encountered in divide
  np.exp(z) / (1 + np.exp(z)))
C:\Users\p528552\AppData\Local\Temp\ipykernel_33268\1788851555.py:12: RuntimeWarning: overflow encountered in exp
  1 / (1 + np.exp(-z)),


=== Walk-Forward Evaluation ===
Accuracy    : 0.5502
Precision   : 0.5699
Recall      : 0.4953
F1 Score    : 0.5300
ROC AUC     : 0.5342
Log Loss    : 13.7507
Confusion Matrix:
[[62 40]
 [54 53]]
